In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("games.csv")

In [ ]:
df["Teams"]

In [ ]:
df = df.sort_values("date")

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
del df["mp.1"]

In [ ]:
del df["mp_opp.1"]

In [ ]:
del df["index_opp"]

In [ ]:
del df["gmsc"]

In [ ]:
df["Teams"]

In [ ]:
del df["+/-"]

In [ ]:
del df["mp_max"]

In [ ]:
df

In [ ]:
df = df.dropna(subset=["target"])

In [ ]:
print(df[["Teams", "date", "won"]].head())

In [ ]:
df["Teams"]

In [ ]:
def add_target(team):
    team["target"] = team["won"].shift(-1)
    return team
df = df.groupby("Teams",group_keys=False).apply(add_target)
    

In [ ]:
df["Teams"]

In [ ]:
df = df.sort_values(["Teams", "date"])



# def add_target(team):
#     team = team.copy()
#     team["target"] = team["won"].shift(-1)
#     return team

# df = df.groupby("Teams", group_keys=False).apply(add_target)

In [ ]:
df["Teams"]

In [ ]:
df["target"] = df.groupby("Teams")["won"].shift(-1)


In [ ]:
df

In [ ]:
df.loc[df["target"].isna(), "target"] = 2

In [ ]:
df["target"] = df["target"].astype(int,errors="ignore")

In [ ]:
df["won"].value_counts()

In [ ]:
df["target"].value_counts()

In [ ]:
nulls = pd.isnull(df)


In [ ]:
nulls = nulls.sum()
nulls
df["Teams"]

In [ ]:
nulls = nulls[nulls>0]


In [ ]:
valid_columns = df.columns[~df.columns.isin(nulls.index)]

In [ ]:
valid_columns["Teams"]

In [ ]:
df["Teams"]

In [ ]:
df

In [ ]:
df = df[valid_columns].copy()


In [ ]:
df.reset_index()

In [ ]:
nulls = pd.isnull(df)


In [ ]:
nulls = nulls.sum()

In [ ]:
nulls[nulls>0]

In [ ]:
df.reset_index()

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import RidgeClassifier

In [ ]:
rr = RidgeClassifier(alpha=1)
split = TimeSeriesSplit(n_splits=3)
sfs = SequentialFeatureSelector(rr, n_features_to_select=30,direction="forward",cv=split)

In [ ]:
removed_columns = ["season","date","won","target","Teams","Teams_opp"]

In [ ]:
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [ ]:
selected_columns

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[selected_columns] = scaler.fit_transform(df[selected_columns])

In [ ]:
df

In [ ]:
sfs.fit(df[selected_columns],df["target"])


In [ ]:
predictors = list(selected_columns[sfs.get_support()])

In [ ]:
predictors

In [ ]:
def backtest (data,model,predictors,start=2,step1=1):
    all_predictions=[]
    seasons = sorted(df["season"].unique())
    for i in range(start,len(seasons),step1):
        season = season[i]
        
        test = data[data["season"] == season]
  
# backtest(df,rr,predictors,2,1)  
seasons = sorted(df["season"].unique())
len(seasons)
# train = df[df["season"] < seasons[1]]
        # model.fit(train[predictors],train["target"])

        # preds = model.predict(test[predictors])
        # preds = pd.se


In [ ]:
df["Teams"]

In [ ]:
split_idx = int(len(df) * 0.8)

In [ ]:
split_idx

In [ ]:
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [ ]:
def prediction(data,model,predictors):
    train_df = data.iloc[:split_idx]
    test_df = data.iloc[split_idx:]
    model.fit(train_df[predictors],train_df["target"])
    preds = model.predict(test_df[predictors])
    preds = pd.Series(preds,index=test_df.index)

    combined = pd.concat([test_df["target"],preds],axis=1)
    combined.columns = ["actual","predictions"]
    return combined

predictions = prediction(df,rr,predictors)

In [ ]:
predictions

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(predictions["actual"],predictions["predictions"])

In [ ]:
df.groupby("home").apply(lambda x: x[x["won"]==1].shape[0] / x.shape[0])

In [ ]:
df["home"]

In [ ]:
print(df["won"].dtype)

In [ ]:
df_rolling = df[list(selected_columns) + ["won","Teams","season"]]

In [ ]:
df_rolling

In [ ]:
def find_team_average(team):
    

In [ ]:
df["Teams"]

In [ ]:
def find_team_averages(team):
    rolling = team.rolling(10).mean()
    return rolling

df_rolling = df_rolling.groupby(["Teams","season"],group_keys=False).apply(find_team_averages)


In [ ]:
df_rolling

In [ ]:
rolling_cols = [f"{col}_10" for col in df_rolling.columns]

In [ ]:
rolling_cols

In [ ]:
df_rolling.columns = rolling_cols

In [ ]:
df = pd.concat([df,df_rolling],axis=1)

In [ ]:
df

In [ ]:
df = df.dropna()

In [ ]:
def next_game(team,col_name):
    next_col = team[col_name].shift(-1)
    return next_col 
def add_col(df,col_name):
    return df.groupby("Teams",group_keys=False).apply(lambda x: next_game(x,col_name))
df["home_next"] = add_col(df,"home")
df["team_opp_next"] = add_col(df,"Teams_opp")
df["date_next"] = add_col(df,"date")

In [ ]:
df

In [ ]:
removed_columns = removed_columns = [
    c for c in df.columns
    if df[c].dtype in ["object", "str", "string"]
] + removed_columns
df.dtypes

In [ ]:
df["team_opp_next"].dtype

In [ ]:
removed_columns

In [ ]:
df.columns[df.dtypes == "object"]

In [ ]:
df

In [ ]:
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [ ]:
selected_columns

In [ ]:
sfs.fit(df[selected_columns],df["target"])

In [ ]:
predictors = list(selected_columns[sfs.get_support()])

In [ ]:
predictors

In [ ]:
predictors = prediction(df,rr,predictors)
predictors

In [ ]:
accuracy_score(predictors["actual"],predictors["predictions"])